Reference

https://www.kaggle.com/code/llkh0a/stanford-rna-3d-folding-part-2-protenix-tbm

In [1]:
import subprocess, os, glob

# Try standard Kaggle dataset paths
for pkg_name, patterns in [
    ('biopython', ['/kaggle/input/biopython-cp312/*.whl', '/kaggle/input/datasets/kami1976/biopython-cp312/*.whl']),
    ('biotite', ['/kaggle/input/biotite/*.whl', '/kaggle/input/datasets/amirrezaaleyasin/biotite/*.whl']),
    ('rdkit', ['/kaggle/input/rdkit-2025-9-5/*.whl', '/kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/*.whl']),
]:
    installed = False
    for pattern in patterns:
        whls = glob.glob(pattern)
        if whls:
            subprocess.run(['pip', 'install', '--no-index', '--no-deps', whls[0]], check=True)
            print(f'Installed {pkg_name} from {whls[0]}')
            installed = True
            break
    if not installed:
        print(f'WARNING: Could not find {pkg_name} wheel, trying pip install...')
        subprocess.run(['pip', 'install', pkg_name], check=False)


Processing /kaggle/input/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Installed biopython from /kaggle/input/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
Installed biotite from /kaggle/input/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl
Installed rdkit from /kaggle/input/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl


In [2]:
import os
import sys
import pandas as pd

# ── Local vs Kaggle mode ─────────────────────────────────────────────────────
# On Kaggle competition rerun, KAGGLE_IS_COMPETITION_RERUN is set to a truthy value.
# When running locally we do NOT exit — instead we cap the test set to a small
# number of samples so the notebook finishes quickly.

IS_KAGGLE = True #bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN", ""))

# How many test samples to use when running locally
LOCAL_N_SAMPLES = None

if IS_KAGGLE:
    print("Running in KAGGLE COMPETITION mode — all test targets will be processed.")
else:
    print(f"Running in LOCAL mode — only the first {LOCAL_N_SAMPLES} test targets "
          f"will be processed to save time.")

Running in KAGGLE COMPETITION mode — all test targets will be processed.


In [3]:
import gc
import json
import os
import time

os.environ["LAYERNORM_TYPE"] = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from Bio.Align import PairwiseAligner
from tqdm import tqdm

In [4]:
def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    # 1. Try atom_array attributes first
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass

    # 2. Fallback to feature dict
    f = data["input_feature_dict"]
    
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
        
    # Heuristic fallback: check which index gives us roughly N_token atoms
    n_tokens = data.get("N_token", torch.tensor(0)).item()
    mask11 = (f["atom_to_tokatom_idx"] == 11).bool()
    mask12 = (f["atom_to_tokatom_idx"] == 12).bool()
    
    c11 = mask11.sum().item()
    c12 = mask12.sum().item()
    
    # Return the one closer to N_tokens (likely one per residue)
    if abs(c11 - n_tokens) < abs(c12 - n_tokens):
        return mask11
    else:
        return mask12

In [5]:

# ─────────────── Paths & Constants ───────────────────────────────────────────
# Auto-detect competition data path
import glob as _glob

DATA_BASE = None
_candidates = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-part-2",
    "/kaggle/input/stanford-rna-3d-folding",
    "/kaggle/input/competitions/stanford-rna-3d-folding-part-2",
    "/kaggle/input/competitions/stanford-rna-3d-folding",
]
for _c in _candidates:
    if os.path.isdir(_c) and os.path.exists(os.path.join(_c, "test_sequences.csv")):
        DATA_BASE = _c
        break

if DATA_BASE is None:
    _matches = _glob.glob("/kaggle/input/**/test_sequences.csv", recursive=True)
    if _matches:
        DATA_BASE = os.path.dirname(_matches[0])
    else:
        print("WARNING: Could not find competition data!")
        print("Available /kaggle/input/ dirs:")
        if os.path.isdir("/kaggle/input"):
            for d in os.listdir("/kaggle/input"):
                full = os.path.join("/kaggle/input", d)
                if os.path.isdir(full):
                    files = os.listdir(full)[:10]
                    print(f"  {d}/: {files}")
        DATA_BASE = "/kaggle/input/competitions/stanford-rna-3d-folding-2"

print(f"DATA_BASE = {DATA_BASE}")
if os.path.isdir(DATA_BASE):
    print(f"  Data files: {os.listdir(DATA_BASE)[:10]}")
else:
    print(f"  WARNING: Data dir does not exist!")

DEFAULT_TEST_CSV       = f"{DATA_BASE}/test_sequences.csv"
DEFAULT_TRAIN_CSV      = f"{DATA_BASE}/train_sequences.csv"
DEFAULT_TRAIN_LBLS     = f"{DATA_BASE}/train_labels.csv"
DEFAULT_VAL_CSV        = f"{DATA_BASE}/validation_sequences.csv"
DEFAULT_VAL_LBLS       = f"{DATA_BASE}/validation_labels.csv"
DEFAULT_OUTPUT         = "/kaggle/working/submission.csv"

# Auto-detect protenix code dir
DEFAULT_CODE_DIR = None
_protenix_candidates = [
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1",
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust/Protenix-v1",
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1",
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted",
    "/kaggle/input/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1",
    "/kaggle/input/protenix-v1-adjusted/Protenix-v1-adjust/Protenix-v1",
    "/kaggle/input/protenix-v1-adjusted/Protenix-v1",
    "/kaggle/input/protenix-v1-adjusted",
    "/kaggle/input/pm-111457844-at-03-09-2026-10-59-40",
]
for _pc in _protenix_candidates:
    if os.path.isdir(_pc):
        has_ckpt = (os.path.isdir(os.path.join(_pc, "checkpoint")) or 
                    os.path.isdir(os.path.join(_pc, "checkpoints")) or
                    len(_glob.glob(os.path.join(_pc, "*.pt"))) > 0 or
                    len(_glob.glob(os.path.join(_pc, "checkpoint", "*.pt"))) > 0)
        has_configs = (os.path.isdir(os.path.join(_pc, "configs")) or
                      os.path.isdir(os.path.join(_pc, "config")))
        has_runner = os.path.isdir(os.path.join(_pc, "runner"))
        if has_ckpt or has_configs or has_runner:
            DEFAULT_CODE_DIR = _pc
            print(f"Found protenix at: {_pc} (ckpt={has_ckpt}, configs={has_configs}, runner={has_runner})")
            break

if DEFAULT_CODE_DIR is None:
    _search_roots = ["/kaggle/input/datasets/*/protenix*", "/kaggle/input/datasets/*/*/protenix*",
                     "/kaggle/input/protenix*", "/kaggle/input/pm-*"]
    _ckpt_matches = []
    for _sr in _search_roots:
        _ckpt_matches.extend(_glob.glob(f"{_sr}/**/checkpoint/*.pt", recursive=True))
        if _ckpt_matches:
            break
    if not _ckpt_matches:
        for _sr in _search_roots:
            _ckpt_matches.extend(_glob.glob(f"{_sr}/**/*.pt", recursive=True))
            if _ckpt_matches:
                break
    if _ckpt_matches:
        _ckpt_dir = os.path.dirname(_ckpt_matches[0])
        if os.path.basename(_ckpt_dir) == "checkpoint":
            DEFAULT_CODE_DIR = os.path.dirname(_ckpt_dir)
        else:
            DEFAULT_CODE_DIR = _ckpt_dir
        print(f"Found protenix via glob: {DEFAULT_CODE_DIR}")
    else:
        print("WARNING: Could not find protenix checkpoint!")
        if os.path.isdir("/kaggle/input"):
            for d in sorted(os.listdir("/kaggle/input")):
                full = os.path.join("/kaggle/input", d)
                if os.path.isdir(full):
                    contents = os.listdir(full)[:10]
                    print(f"  {d}/: {contents}")
                    for sub in os.listdir(full)[:5]:
                        subfull = os.path.join(full, sub)
                        if os.path.isdir(subfull):
                            subcontents = os.listdir(subfull)[:8]
                            print(f"    {sub}/: {subcontents}")
        DEFAULT_CODE_DIR = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"

print(f"DEFAULT_CODE_DIR = {DEFAULT_CODE_DIR}")
if os.path.isdir(DEFAULT_CODE_DIR):
    print(f"  Contents: {os.listdir(DEFAULT_CODE_DIR)[:15]}")
else:
    print(f"  WARNING: Directory does not exist!")
DEFAULT_ROOT_DIR = DEFAULT_CODE_DIR

# Symlink setup for Protenix
if DEFAULT_CODE_DIR and os.path.isdir(DEFAULT_CODE_DIR):
    _protenix_root = Path(DEFAULT_CODE_DIR)
    _mmcif_candidates = [
        _protenix_root / "common",
        _protenix_root.parent / "common",
        Path("/kaggle/input/protenix-v1-adjusted") / "common",
    ]
    for _mc in _mmcif_candidates:
        if _mc.is_dir() and list(_mc.glob("*.cif"))[:1]:
            if not (_protenix_root / "common").exists():
                os.symlink(str(_mc), str(_protenix_root / "common"))
                print(f"Symlinked common/ -> {_mc}")
            break

MODEL_NAME    = "protenix_base_20250630_v1.0.0"
N_SAMPLE      = 5
SEED          = 42
MAX_SEQ_LEN   = int(os.environ.get("MAX_SEQ_LEN",   "512"))
# FIX 1: CHUNK_OVERLAP 128 -> 256 (top solution uses 256, better stitching)
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP",  "256"))

# TBM quality thresholds — lowered to capture more templates
MIN_SIMILARITY       = float(os.environ.get("MIN_SIMILARITY",       "0.0"))
# FIX 3: MIN_PERCENT_IDENTITY 50 -> 40 (capture more template matches)
MIN_PERCENT_IDENTITY = float(os.environ.get("MIN_PERCENT_IDENTITY", "40.0"))

USE_PROTENIX = True


def parse_bool(value: str, default: bool = False) -> str:
    v = str(value).strip().lower()
    if v in {"1", "true", "t", "yes", "y", "on"}:
        return "true"
    if v in {"0", "false", "f", "no", "n", "off"}:
        return "false"
    return "true" if default else "false"


USE_MSA      = parse_bool(os.environ.get("USE_MSA",      "false"))
# Templates disabled - requires mmcif directory not available on Kaggle
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA  = parse_bool(os.environ.get("USE_RNA_MSA",  "true"))

MODEL_N_SAMPLE = int(os.environ.get("MODEL_N_SAMPLE", str(N_SAMPLE)))


# ─────────────── General Utilities ───────────────────────────────────────────
def seed_everything(seed: int) -> None:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)


def resolve_paths():
    test_csv   = os.environ.get("TEST_CSV",           DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV",     DEFAULT_OUTPUT)
    code_dir   = os.environ.get("PROTENIX_CODE_DIR",  DEFAULT_CODE_DIR)
    root_dir   = os.environ.get("PROTENIX_ROOT_DIR",  DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir


def ensure_required_files(root_dir: str) -> None:
    for p, name in [
        (Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt",          "checkpoint"),
        (Path(root_dir) / "common" / "components.cif",                "CCD file"),
        (Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl",  "CCD cache"),
    ]:
        if not p.exists():
            raise FileNotFoundError(f"Missing {name}: {p}")


# ─────────────── Protenix Input / Config Helpers ─────────────────────────────
def build_input_json(df: pd.DataFrame, json_path: str) -> None:
    data = [
        {
            "name": row["target_id"],
            "covalent_bonds": [],
            "sequences": [{"rnaSequence": {"sequence": row["sequence"], "count": 1}}],
        }
        for _, row in df.iterrows()
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(t, p):
        for k, v in p.items():
            if isinstance(v, dict) and k in t and isinstance(t[k], dict):
                deep_update(t[k], v)
            else:
                t[k] = v

    deep_update(base, model_configs[model_name])
    arg_str = " ".join([
        f"--model_name {model_name}",
        f"--input_json_path {input_json_path}",
        f"--dump_dir {dump_dir}",
        f"--use_msa {USE_MSA}",
        f"--use_template {USE_TEMPLATE}",
        f"--use_rna_msa {USE_RNA_MSA}",
        f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
        f"--seeds {SEED}",
    ])
    return parse_configs(configs=base, arg_str=arg_str, fill_required_with_null=True)


def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"):
                    base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except Exception:
            pass
    f = data["input_feature_dict"]
    if "center_atom_mask" in f:
        return (f["center_atom_mask"] == 1).bool()
    if "centre_atom_mask" in f:
        return (f["centre_atom_mask"] == 1).bool()
    return (f["atom_to_tokatom_idx"] == 11).bool()


def get_feature_c1_mask(data: dict) -> torch.Tensor:
    f = data["input_feature_dict"]
    if "centre_atom_mask" in f:
        return f["centre_atom_mask"].long() == 1
    return f["atom_to_tokatom_idx"].long() == 12


def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list:
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i + 1}", "resname": seq[i], "resid": i + 1}
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows


def pad_samples(coords: np.ndarray, n: int) -> np.ndarray:
    if coords.shape[0] >= n:
        return coords[:n]
    if coords.shape[0] == 0:
        return np.zeros((n, coords.shape[1], 3), dtype=coords.dtype)
    extra = np.repeat(coords[:1], n - coords.shape[0], axis=0)
    return np.concatenate([coords, extra], axis=0)


def split_into_chunks(seq_len: int, max_len: int, overlap: int) -> list:
    if seq_len <= max_len:
        return [(0, seq_len)]
    chunks = []
    step = max_len - overlap
    pos = 0
    while pos < seq_len:
        end = min(pos + max_len, seq_len)
        chunks.append((pos, end))
        if end == seq_len:
            break
        pos += step
    return chunks


def kabsch_align(P: np.ndarray, Q: np.ndarray):
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    Pc = P - centroid_P
    Qc = Q - centroid_Q
    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    S = np.eye(3)
    if d < 0:
        S[2, 2] = -1
    R = Vt.T @ S @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t


def stitch_chunk_coords(chunk_coords_list, chunk_ranges, seq_len):
    if len(chunk_coords_list) == 1:
        coords = chunk_coords_list[0]
        if coords.shape[0] >= seq_len:
            return coords[:seq_len]
        out = np.zeros((seq_len, 3), dtype=coords.dtype)
        out[:coords.shape[0]] = coords
        return out
    aligned = [chunk_coords_list[0].copy()]
    for i in range(1, len(chunk_coords_list)):
        prev_start, prev_end = chunk_ranges[i - 1]
        cur_start, cur_end = chunk_ranges[i]
        ov_start = cur_start
        ov_end = min(prev_end, cur_end)
        ov_len = ov_end - ov_start
        if ov_len < 3:
            aligned.append(chunk_coords_list[i].copy())
            continue
        prev_ov = aligned[i - 1][ov_start - prev_start: ov_end - prev_start]
        cur_ov = chunk_coords_list[i][ov_start - cur_start: ov_end - cur_start]
        valid = ~(np.isnan(prev_ov).any(axis=1) | np.isnan(cur_ov).any(axis=1))
        if valid.sum() < 3:
            aligned.append(chunk_coords_list[i].copy())
            continue
        R, t = kabsch_align(cur_ov[valid], prev_ov[valid])
        transformed = (chunk_coords_list[i] @ R.T) + t
        aligned.append(transformed)
    full = np.zeros((seq_len, 3), dtype=np.float64)
    weights = np.zeros(seq_len, dtype=np.float64)
    for i, ((s, e), coords) in enumerate(zip(chunk_ranges, aligned)):
        chunk_len = coords.shape[0]
        actual_end = min(s + chunk_len, seq_len)
        used_len = actual_end - s
        w = np.ones(used_len, dtype=np.float64)
        if i > 0:
            ov_start = s
            ov_end = min(chunk_ranges[i - 1][1], e)
            ramp_len = ov_end - ov_start
            if ramp_len > 0:
                w[:ramp_len] = np.linspace(0.0, 1.0, ramp_len)
        if i < len(chunk_ranges) - 1:
            next_s = chunk_ranges[i + 1][0]
            ramp_start = next_s - s
            ramp_len = actual_end - next_s
            if ramp_len > 0 and ramp_start < used_len:
                w[ramp_start:used_len] = np.linspace(1.0, 0.0, ramp_len)
        full[s:actual_end] += coords[:used_len] * w[:, None]
        weights[s:actual_end] += w
    mask = weights > 0
    full[mask] /= weights[mask, None]
    return full


def _make_aligner():
    al = PairwiseAligner()
    al.mode = "global"
    al.match_score = 2; al.mismatch_score = -1.5
    al.open_gap_score = -8; al.extend_gap_score = -0.3  # relaxed from -0.4
    al.query_left_open_gap_score = -8; al.query_left_extend_gap_score = -0.3
    al.query_right_open_gap_score = -8; al.query_right_extend_gap_score = -0.3
    al.target_left_open_gap_score = -8; al.target_left_extend_gap_score = -0.3
    al.target_right_open_gap_score = -8; al.target_right_extend_gap_score = -0.3
    return al

_aligner = _make_aligner()


def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(";") for ch, cnt in [part.split(":")]]


def parse_fasta(fasta_content):
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line: continue
        if line.startswith(">"):
            if cur is not None: out[cur] = "".join(parts)
            cur = line[1:].split()[0]; parts = []
        else:
            parts.append(line.replace(" ", ""))
    if cur is not None: out[cur] = "".join(parts)
    return out


def get_chain_segments(row):
    seq = row["sequence"]; stoich = row.get("stoichiometry", ""); all_sq = row.get("all_sequences", "")
    if pd.isna(stoich) or pd.isna(all_sq) or str(stoich).strip() == "" or str(all_sq).strip() == "":
        return [(0, len(seq))]
    try:
        chain_dict = parse_fasta(all_sq); order = parse_stoichiometry(stoich)
        segs, pos = [], 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None: return [(0, len(seq))]
            for _ in range(cnt): segs.append((pos, pos + len(base))); pos += len(base)
        return segs if pos == len(seq) else [(0, len(seq))]
    except Exception:
        return [(0, len(seq))]


def build_segments_map(df):
    seg_map, stoich_map = {}, {}
    for _, r in df.iterrows():
        tid = r["target_id"]; seg_map[tid] = get_chain_segments(r)
        raw_s = r.get("stoichiometry", ""); stoich_map[tid] = "" if pd.isna(raw_s) else str(raw_s)
    return seg_map, stoich_map


def process_labels(labels_df):
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1", "y_1", "z_1"]].values
    return coords


def _build_aligned_strings(query_seq, template_seq, alignment):
    q_segs, t_segs = alignment.aligned
    aq, at, qi, ti = [], [], 0, 0
    for (qs, qe), (ts, te) in zip(q_segs, t_segs):
        while qi < qs: aq.append(query_seq[qi]); at.append("-"); qi += 1
        while ti < ts: aq.append("-"); at.append(template_seq[ti]); ti += 1
        for qp, tp in zip(range(qs, qe), range(ts, te)):
            aq.append(query_seq[qp]); at.append(template_seq[tp])
        qi, ti = qe, te
    while qi < len(query_seq): aq.append(query_seq[qi]); at.append("-"); qi += 1
    while ti < len(template_seq): aq.append("-"); at.append(template_seq[ti]); ti += 1
    return "".join(aq), "".join(at)


def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=30):
    results = []
    for _, row in train_seqs_df.iterrows():
        tid, tseq = row["target_id"], row["sequence"]
        if tid not in train_coords_dict: continue
        if abs(len(tseq) - len(query_seq)) / max(len(tseq), len(query_seq)) > 0.3: continue
        aln = next(iter(_aligner.align(query_seq, tseq)))
        norm_s = aln.score / (2 * min(len(query_seq), len(tseq)))
        identical = sum(1 for (qs, qe), (ts, te) in zip(*aln.aligned)
                        for qp, tp in zip(range(qs, qe), range(ts, te)) if query_seq[qp] == tseq[tp])
        pct_id = 100 * identical / len(query_seq)
        aq, at = _build_aligned_strings(query_seq, tseq, aln)
        results.append((tid, tseq, norm_s, train_coords_dict[tid], pct_id, aq, at))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:top_n]


def adapt_template_to_query(query_seq, template_seq, template_coords):
    aln = next(iter(_aligner.align(query_seq, template_seq)))
    new_coords = np.full((len(query_seq), 3), np.nan)
    for (qs, qe), (ts, te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk) == (qe - qs): new_coords[qs:qe] = chunk
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            pv = next((j for j in range(i - 1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            nv = next((j for j in range(i + 1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)
            if pv >= 0 and nv >= 0:
                w = (i - pv) / (nv - pv); new_coords[i] = (1 - w) * new_coords[pv] + w * new_coords[nv]
            elif pv >= 0: new_coords[i] = new_coords[pv] + [3, 0, 0]
            elif nv >= 0: new_coords[i] = new_coords[nv] + [3, 0, 0]
            else: new_coords[i] = [i * 3, 0, 0]
    return np.nan_to_num(new_coords)


# FIX 2: RNA constraint passes 2 -> 3 (top solution uses 3, more physically realistic)
def adaptive_rna_constraints(coords, target_id, segments_map, confidence=1.0, passes=3):
    X = coords.copy(); segments = segments_map.get(target_id, [(0, len(X))])
    strength = max(0.75 * (1.0 - min(confidence, 0.97)), 0.02)
    for _ in range(passes):
        for s, e in segments:
            C = X[s:e]; L = e - s
            if L < 3: continue
            d = C[1:] - C[:-1]; dist = np.linalg.norm(d, axis=1) + 1e-6
            adj = d * ((5.95 - dist) / dist)[:, None] * (0.22 * strength); C[:-1] -= adj; C[1:] += adj
            d2 = C[2:] - C[:-2]; d2n = np.linalg.norm(d2, axis=1) + 1e-6
            adj2 = d2 * ((10.2 - d2n) / d2n)[:, None] * (0.10 * strength); C[:-2] -= adj2; C[2:] += adj2
            C[1:-1] += (0.06 * strength) * (0.5 * (C[:-2] + C[2:]) - C[1:-1])
            if L >= 25:
                idx = np.linspace(0, L - 1, min(L, 160)).astype(int) if L > 220 else np.arange(L)
                P = C[idx]; diff = P[:, None, :] - P[None, :, :]
                dm = np.linalg.norm(diff, axis=2) + 1e-6; sep = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (dm < 3.2)
                if np.any(mask):
                    vec = (diff * ((3.2 - dm) / dm)[:, :, None] * mask[:, :, None]).sum(axis=1)
                    C[idx] += (0.015 * strength) * vec
            X[s:e] = C
    return X


def _rotmat(axis, ang):
    a = np.asarray(axis, float); a /= np.linalg.norm(a) + 1e-12
    x, y, z = a; c, s = np.cos(ang), np.sin(ang); CC = 1 - c
    return np.array([[c+x*x*CC, x*y*CC-z*s, x*z*CC+y*s],
                     [y*x*CC+z*s, c+y*y*CC, y*z*CC-x*s],
                     [z*x*CC-y*s, z*y*CC+x*s, c+z*z*CC]])


def apply_hinge(coords, seg, rng, deg=22):
    s, e = seg; L = e - s
    if L < 30: return coords
    pivot = s + int(rng.integers(10, L - 10))
    R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
    X = coords.copy(); p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X


def jitter_chains(coords, segs, rng, deg=12, trans=1.5):
    X = coords.copy(); gc_ = X.mean(0, keepdims=True)
    for s, e in segs:
        R = _rotmat(rng.normal(size=3), np.deg2rad(float(rng.uniform(-deg, deg))))
        shift = rng.normal(size=3); shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0, trans))
        c = X[s:e].mean(0, keepdims=True); X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(0, keepdims=True) - gc_
    return X


def smooth_wiggle(coords, segs, rng, amp=0.8):
    X = coords.copy()
    for s, e in segs:
        L = e - s
        if L < 20: continue
        ctrl = np.linspace(0, L - 1, 6); disp = rng.normal(0, amp, (6, 3)); t = np.arange(L)
        X[s:e] += np.vstack([np.interp(t, ctrl, disp[:, k]) for k in range(3)]).T
    return X


def generate_rna_structure(sequence, seed=None):
    if seed is not None: np.random.seed(seed)
    n = len(sequence); coords = np.zeros((n, 3))
    for i in range(n):
        ang = i * 0.6; coords[i] = [10.0 * np.cos(ang), 10.0 * np.sin(ang), i * 2.5]
    return coords


def tbm_phase(test_df, train_seqs_df, train_coords_dict, segments_map):
    print(f"\n{'='*60}\nPHASE 1: Template-Based Modeling")
    print(f"  MIN_SIMILARITY = {MIN_SIMILARITY}  |  MIN_PCT_IDENTITY = {MIN_PERCENT_IDENTITY}\n{'='*60}")
    t0 = time.time()
    template_predictions, protenix_queue = {}, {}
    for _, row in test_df.iterrows():
        tid, seq = row["target_id"], row["sequence"]
        segs = segments_map.get(tid, [(0, len(seq))])
        similar = find_similar_sequences_detailed(seq, train_seqs_df, train_coords_dict, top_n=30)
        preds, used = [], set()
        for i, (tmpl_id, tmpl_seq, sim, tmpl_coords, pct_id, _, _) in enumerate(similar):
            if len(preds) >= N_SAMPLE: break
            if sim < MIN_SIMILARITY or pct_id < MIN_PERCENT_IDENTITY: break
            if tmpl_id in used: continue
            rng = np.random.default_rng((row.name * 10000000000 + i * 10007) % (2**32))
            adapted = adapt_template_to_query(seq, tmpl_seq, tmpl_coords)
            slot = len(preds)
            if slot == 0: X = adapted
            elif slot == 1: X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
            elif slot == 2:
                longest = max(segs, key=lambda se: se[1] - se[0]); X = apply_hinge(adapted, longest, rng)
            elif slot == 3: X = jitter_chains(adapted, segs, rng)
            else: X = smooth_wiggle(adapted, segs, rng)
            refined = adaptive_rna_constraints(X, tid, segments_map, confidence=sim)
            preds.append(refined); used.add(tmpl_id)
        template_predictions[tid] = preds
        n_needed = N_SAMPLE - len(preds)
        if n_needed > 0:
            protenix_queue[tid] = (n_needed, seq)
            print(f"  {tid} ({len(seq)} nt): {len(preds)} TBM -> need {n_needed} from Protenix")
        else:
            print(f"  {tid} ({len(seq)} nt): all {N_SAMPLE} from TBM")
    elapsed = time.time() - t0
    n_full = len(test_df) - len(protenix_queue)
    print(f"\nPhase 1 done in {elapsed:.1f}s\n  Fully covered by TBM: {n_full}\n  Need Protenix: {len(protenix_queue)}")
    return template_predictions, protenix_queue


def main():
    test_csv, output_csv, code_dir, root_dir = resolve_paths()
    if not os.path.isdir(code_dir):
        raise FileNotFoundError(f"Missing PROTENIX_CODE_DIR: {code_dir}. Set PROTENIX_CODE_DIR to the repo path.")
    os.environ["PROTENIX_ROOT_DIR"] = root_dir
    sys.path.append(code_dir)
    ensure_required_files(root_dir)
    seed_everything(SEED)
    test_df_full = pd.read_csv(test_csv)
    test_df = (test_df_full.head(LOCAL_N_SAMPLES) if not IS_KAGGLE else test_df_full).reset_index(drop=True)
    print(f"Test targets: {len(test_df)}" + (" (LOCAL MODE)" if not IS_KAGGLE else ""))
    seq_by_id = dict(zip(test_df["target_id"], test_df["sequence"]))
    test_df_trunc = test_df.copy()
    test_df_trunc["sequence"] = test_df_trunc["sequence"].str[:MAX_SEQ_LEN]
    print("\nLoading training data for TBM ...")
    train_seqs = pd.read_csv(DEFAULT_TRAIN_CSV)
    val_seqs = pd.read_csv(DEFAULT_VAL_CSV)
    train_labels = pd.read_csv(DEFAULT_TRAIN_LBLS)
    val_labels = pd.read_csv(DEFAULT_VAL_LBLS)
    combined_seqs = pd.concat([train_seqs, val_seqs], ignore_index=True)
    combined_labels = pd.concat([train_labels, val_labels], ignore_index=True)
    train_coords = process_labels(combined_labels)
    segments_map, _ = build_segments_map(test_df)
    print(f"Template pool: {len(combined_seqs)} sequences, {len(train_coords)} structures")
    template_preds, protenix_queue = tbm_phase(test_df, combined_seqs, train_coords, segments_map)
    protenix_preds = {}
    if protenix_queue and USE_PROTENIX:
        print(f"\n{'='*60}\nPHASE 2: Protenix for {len(protenix_queue)} targets\n{'='*60}")
        work_dir = Path("/kaggle/working"); work_dir.mkdir(parents=True, exist_ok=True)
        tasks, chunk_info = [], {}
        for target_id, (n_needed, full_seq) in protenix_queue.items():
            seq_len = len(full_seq)
            if seq_len <= MAX_SEQ_LEN:
                tasks.append({"target_id": target_id, "sequence": full_seq})
                chunk_info[target_id] = [{"name": target_id, "range": (0, seq_len)}]
            else:
                chunks = split_into_chunks(seq_len, MAX_SEQ_LEN, CHUNK_OVERLAP)
                chunk_info[target_id] = []
                for ci, (cs, ce) in enumerate(chunks):
                    chunk_name = f"{target_id}_chunk{ci}"
                    tasks.append({"target_id": chunk_name, "sequence": full_seq[cs:ce]})
                    chunk_info[target_id].append({"name": chunk_name, "range": (cs, ce)})
        tasks_df = pd.DataFrame(tasks)
        input_json_path = str(work_dir / "protenix_queue_input.json")
        build_input_json(tasks_df, input_json_path)
        from protenix.data.inference.infer_dataloader import InferenceDataset
        from runner.inference import InferenceRunner, update_gpu_compatible_configs, update_inference_configs
        configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
        configs = update_gpu_compatible_configs(configs)
        runner = InferenceRunner(configs)
        dataset = InferenceDataset(configs)
        raw_predictions = {}
        def _extract_c1_coords(prediction, feat, chunk_seq_len, raw_coords):
            if "centre_atom_mask" in feat:
                mask = (feat["centre_atom_mask"] == 1).to(raw_coords.device)
            elif "atom_to_tokatom_idx" in feat:
                m11 = (feat["atom_to_tokatom_idx"] == 11).to(raw_coords.device)
                m12 = (feat["atom_to_tokatom_idx"] == 12).to(raw_coords.device)
                mask = m11 if abs(m11.sum() - chunk_seq_len) < abs(m12.sum() - chunk_seq_len) else m12
            else:
                mask = torch.zeros(raw_coords.shape[1], dtype=torch.bool, device=raw_coords.device)
            coords = raw_coords[:, mask, :].detach().cpu().numpy()
            if coords.shape[1] > 1:
                diffs = np.linalg.norm(coords[0, 1:] - coords[0, :-1], axis=-1)
                if np.all(diffs < 1e-4): return None
            if coords.shape[1] != chunk_seq_len:
                if coords.shape[1] == 1 and chunk_seq_len > 1: return None
                padded = np.zeros((coords.shape[0], chunk_seq_len, 3), dtype=np.float32)
                ml = min(coords.shape[1], chunk_seq_len); padded[:, :ml, :] = coords[:, :ml, :]; coords = padded
            return coords
        for i in tqdm(range(len(dataset)), desc="Protenix Inference"):
            data, atom_array, err = dataset[i]
            sample_name = data.get("sample_name", f"sample_{i}")
            if err:
                raw_predictions[sample_name] = None
                del data, atom_array, err; gc.collect(); torch.cuda.empty_cache(); continue
            target_id = sample_name.split("_chunk")[0] if "_chunk" in sample_name else sample_name
            n_needed = protenix_queue.get(target_id, (N_SAMPLE, ""))[0]
            sub_seq_len = data["N_token"].item()
            try:
                new_cfg = update_inference_configs(configs, sub_seq_len)
                new_cfg.sample_diffusion.N_sample = n_needed
                runner.update_model_configs(new_cfg)
                pred = runner.predict(data); raw_coords = pred["coordinate"]
                coords = _extract_c1_coords(pred, data["input_feature_dict"], sub_seq_len, raw_coords)
                raw_predictions[sample_name] = coords
            except Exception as exc:
                print(f"  {sample_name} inference failed: {exc}")
                import traceback; traceback.print_exc()
                raw_predictions[sample_name] = None
            finally:
                try: del pred, data, atom_array, raw_coords
                except: pass
                gc.collect(); torch.cuda.empty_cache()
        for target_id, (n_needed, full_seq) in protenix_queue.items():
            seq_len = len(full_seq); chunks = chunk_info.get(target_id, [])
            if not chunks: continue
            if len(chunks) == 1:
                protenix_preds[target_id] = raw_predictions.get(target_id)
            else:
                chunk_results_per_sample = {s: [] for s in range(n_needed)}; all_ok = True
                for cinfo in chunks:
                    ccoords = raw_predictions.get(cinfo["name"])
                    if ccoords is None: all_ok = False; break
                    for s_idx in range(n_needed):
                        chunk_results_per_sample[s_idx].append(
                            (ccoords[s_idx] if s_idx < ccoords.shape[0] else ccoords[-1], cinfo["range"]))
                if not all_ok: protenix_preds[target_id] = None; continue
                stitched = []
                for s_idx in range(n_needed):
                    items = chunk_results_per_sample[s_idx]
                    stitched.append(stitch_chunk_coords([c for c, _ in items], [r for _, r in items], seq_len))
                protenix_preds[target_id] = np.stack(stitched, axis=0)
    elif protenix_queue and not USE_PROTENIX:
        print(f"\nPHASE 2 skipped. De-novo fallback for {len(protenix_queue)} targets.")
    print(f"\n{'='*60}\nPHASE 3: Combine TBM + Protenix + de-novo fallback\n{'='*60}")
    all_rows = []
    for _, row in test_df.iterrows():
        tid, seq = row["target_id"], row["sequence"]
        combined = list(template_preds.get(tid, []))
        ptx = protenix_preds.get(tid)
        if ptx is not None and ptx.ndim == 3:
            for j in range(ptx.shape[0]):
                if len(combined) >= N_SAMPLE: break
                combined.append(ptx[j])
        while len(combined) < N_SAMPLE:
            seed_val = row.name * 1000000 + len(combined) * 1000
            dn = generate_rna_structure(seq, seed=seed_val)
            combined.append(adaptive_rna_constraints(dn, tid, segments_map, confidence=0.2))
        stacked = np.stack(combined[:N_SAMPLE], axis=0)
        all_rows.extend(coords_to_rows(tid, seq, stacked))
    sub = pd.DataFrame(all_rows)
    cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
    sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
    sub[cols].to_csv(output_csv, index=False)
    print(f"\nSaved submission to {output_csv}  ({len(sub):,} rows)")

DATA_BASE = /kaggle/input/competitions/stanford-rna-3d-folding-2
  Data files: ['MSA', 'sample_submission.csv', 'validation_sequences.csv', 'test_sequences.csv', 'validation_labels.csv', 'extra', 'train_labels.csv', 'train_sequences.csv', 'PDB_RNA']
Found protenix at: /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1 (ckpt=True, configs=True, runner=True)
DEFAULT_CODE_DIR = /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1
  Contents: ['CONTRIBUTING.md', 'CODE_OF_CONDUCT.md', 'tests', 'Dockerfile', 'assets', '.pre-commit-config.yaml', 'scripts', 'configs', 'inference_demo.sh', 'LICENSE', '.gitignore', 'examples', '.github', 'README.md', 'docs']


/usr/local/lib/python3.12/dist-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_left_open_gap_score' was renamed to 'open_left_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_left_extend_gap_score' was renamed to 'extend_left_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_right_open_gap_score' was renamed to 'open_right_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio

# Main

In [6]:
import traceback as _tb

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"\n{'='*60}")
        print(f"MAIN FAILED: {e}")
        _tb.print_exc()
        print(f"{'='*60}")
        print("Creating fallback dummy submission...")
        
        # Try to read test CSV to get proper target IDs
        import pandas as _pd
        _output = "/kaggle/working/submission.csv"
        _test_csv = None
        for _tc in [
            "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv",
            "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv",
            "/kaggle/input/stanford-rna-3d-folding-part-2/test_sequences.csv",
        ]:
            if os.path.exists(_tc):
                _test_csv = _tc
                break
        
        if _test_csv is None:
            import glob as _g
            _matches = _g.glob("/kaggle/input/**/test_sequences.csv", recursive=True)
            if _matches:
                _test_csv = _matches[0]
        
        _rows = []
        if _test_csv and os.path.exists(_test_csv):
            _tdf = _pd.read_csv(_test_csv)
            for _, _r in _tdf.iterrows():
                _tid = _r["target_id"]
                _seq = _r["sequence"]
                for _i in range(len(_seq)):
                    _row = {"ID": f"{_tid}_{_i+1}", "resname": _seq[_i], "resid": _i+1}
                    for _s in range(1, 6):
                        _ang = _i * 0.6
                        _row[f"x_{_s}"] = float(10.0 * np.cos(_ang) + (_s-1)*0.1)
                        _row[f"y_{_s}"] = float(10.0 * np.sin(_ang) + (_s-1)*0.1)
                        _row[f"z_{_s}"] = float(_i * 2.5 + (_s-1)*0.1)
                    _rows.append(_row)
        else:
            # Absolute last resort - create minimal valid submission
            for _i in range(10):
                _row = {"ID": f"dummy_{_i+1}", "resname": "A", "resid": _i+1}
                for _s in range(1, 6):
                    _row[f"x_{_s}"] = float(_i)
                    _row[f"y_{_s}"] = 0.0
                    _row[f"z_{_s}"] = 0.0
                _rows.append(_row)
        
        _sub = _pd.DataFrame(_rows)
        _cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, 6) for c in ["x", "y", "z"]]
        _sub[[c for c in _cols if c in _sub.columns]].to_csv(_output, index=False)
        print(f"Fallback submission saved: {_output} ({len(_sub)} rows)")

Test targets: 28

Loading training data for TBM ...


/tmp/ipykernel_24/1500712931.py:618: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  train_labels = pd.read_csv(DEFAULT_TRAIN_LBLS)


Template pool: 5744 sequences, 5744 structures

PHASE 1: Template-Based Modeling
  MIN_SIMILARITY = 0.0  |  MIN_PCT_IDENTITY = 40.0
  8ZNQ (30 nt): all 5 from TBM
  9IWF (69 nt): all 5 from TBM
  9JGM (210 nt): all 5 from TBM
  9MME (4640 nt): all 5 from TBM
  9J09 (214 nt): all 5 from TBM
  9E9Q (101 nt): all 5 from TBM
  9CFN (59 nt): all 5 from TBM
  9OBM (73 nt): all 5 from TBM
  9G4P (68 nt): all 5 from TBM
  9G4Q (104 nt): all 5 from TBM
  9G4R (47 nt): all 5 from TBM
  9RVP (34 nt): all 5 from TBM
  9JFS (246 nt): 3 TBM -> need 2 from Protenix
  9LEC (378 nt): all 5 from TBM
  9LEL (476 nt): all 5 from TBM
  9I9W (28 nt): all 5 from TBM
  9HRO (35 nt): all 5 from TBM
  9QZJ (19 nt): all 5 from TBM
  9JFO (195 nt): all 5 from TBM
  9OD4 (23 nt): all 5 from TBM
  9WHV (80 nt): all 5 from TBM
  9E74 (255 nt): all 5 from TBM
  9E75 (165 nt): all 5 from TBM
  9G4J (334 nt): all 5 from TBM
  9KGG (267 nt): all 5 from TBM
  9EBP (81 nt): all 5 from TBM
  9LJN (71 nt): all 5 from TBM
  

2026-03-22 19:01:56,418 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Distributed environment: world size: 1, global rank: 0, local rank: 0
2026-03-22 19:01:56,419 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:98] INFO root: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
2026-03-22 19:01:56,547 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:127] INFO root: Finished environment initialization.


train scheduler 16.0
inference scheduler 16.0
Diffusion Module has 16.0


2026-03-22 19:03:13,926 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Loading from /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/checkpoint/protenix_base_20250630_v1.0.0.pt, strict: True
2026-03-22 19:03:21,468 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Sampled key: module.input_embedder.atom_attention_encoder.linear_no_bias_ref_pos.weight
2026-03-22 19:03:21,616 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Finish loading checkpoint.
2026-03-22 19:03:21,628 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] 


PHASE 3: Combine TBM + Protenix + de-novo fallback

Saved submission to /kaggle/working/submission.csv  (9,762 rows)


In [7]:
#read submission.csv
submission_path = "/kaggle/working/submission.csv"
submission_df = pd.read_csv(submission_path)
print(submission_df.head(20))

         ID resname  resid        x_1        y_1        z_1         x_2  \
0    8ZNQ_1       A      1  -2.054368 -15.061686  20.742231  145.471750   
1    8ZNQ_2       C      2  -1.971488 -15.075482  15.339274  141.460490   
2    8ZNQ_3       C      3  -3.347056 -13.501452  10.451633  138.154297   
3    8ZNQ_4       G      4  -5.445126 -11.037820   6.600842  142.474618   
4    8ZNQ_5       U      5  -6.349440  -6.252547   4.542190  143.578173   
5    8ZNQ_6       G      6  -6.914110  -0.883021   3.236031  142.571399   
6    8ZNQ_7       A      7  -4.499149   4.119476  -0.901018  140.223242   
7    8ZNQ_8       C      8  -5.397797   9.769351   3.347287  137.362748   
8    8ZNQ_9       G      9  -0.397137  10.593469   0.117009  135.640290   
9   8ZNQ_10       G     10   3.904360   9.965095  -2.053605  134.943596   
10  8ZNQ_11       G     11   7.969742   7.516582  -4.496067  133.474305   
11  8ZNQ_12       C     12  10.184632   4.492326  -8.999271  131.995465   
12  8ZNQ_13       C     1